In [1]:
from src.rag.repo_rag import RepositoryRAG
from src.rag.config import PipelineConfig
from src.utils.qdrant_store import QdrantStore
from src.utils.retrieval import KnowledgeGraphRetriever
from src.utils.deduplicator import Deduplicator
from src.utils.reranker import Reranker
from typing import Tuple
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline


from huggingface_hub import login

def build_llm(
    llm_model: str,
    quantize: bool = False,
    use_4bit: bool = True,
    bnb_4bit_use_double_quant: bool = True,
    bnb_4bit_quant_type: str = "nf4",
    bnb_4bit_compute_dtype = torch.bfloat16,
    use_8bit: bool = False,
) -> Tuple[AutoModelForCausalLM, AutoTokenizer, pipeline]:
    """
    Build (optionally quantized) LLM + tokenizer + generation pipeline.

    Args:
      llm_model: HF model id
      quantize: whether to load a quantized model
      use_4bit: if quantize=True, prefer 4-bit (nf4) quantization. If False and quantize=True, will try 8-bit
      bnb_4bit_*: bitsandbytes config options for 4-bit
      use_8bit: explicit 8-bit flag (overrides use_4bit when quantize=True)

    Returns:
      (model, tokenizer, gen_pipeline)
    """

    # tokenizer (safe to always load)
    tokenizer = AutoTokenizer.from_pretrained(llm_model, padding_side="left", use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # decide device availability
    has_cuda = torch.cuda.is_available()
    if quantize and not has_cuda:
        raise RuntimeError("Quantization (bitsandbytes) requires CUDA. Set quantize=False or run on GPU.")

    model = None

    if quantize:
        # prefer explicit 4-bit if use_4bit True and use_8bit False
        if use_4bit and not use_8bit:
            # 4-bit config using BitsAndBytesConfig
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=bnb_4bit_use_double_quant,
                bnb_4bit_quant_type=bnb_4bit_quant_type,
                bnb_4bit_compute_dtype=bnb_4bit_compute_dtype,
            )
        
        else:
            # fallback to 8-bit (older but widely supported)
            bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            llm_model,
            device_map="auto",
            quantization_config=bnb_config,
            torch_dtype=torch.float16
        )

    else:
        # Full precision or mixed precision (let transformers pick optimal device_map)
        # If CUDA present, we request float16 for speed; otherwise default dtype.
        if has_cuda:
            model = AutoModelForCausalLM.from_pretrained(llm_model, device_map="auto", torch_dtype=torch.float16)
        else:
            # CPU fallback (may be slow)
            model = AutoModelForCausalLM.from_pretrained(llm_model, device_map="auto")

    # Build generation pipeline. We pass the already loaded model + tokenizer.
    gen = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        device_map="auto"  # keep this consistent with model device_map
    )

    return model, tokenizer, gen


d:\EricssonCodeGraph\hgb-rag-cqa\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
token_path ="./_/hf_token.txt"
with open(token_path, "r") as f:
    huggingface_apikey = f.read().strip()

login(huggingface_apikey)

In [6]:
# Configuration
config = PipelineConfig(
    verbose=False,
    retriever="kg",
    top_k=10,
    llm_max_tokens=200,
    deduplicate=True,
    dedup_use_minhash=True,
    dedup_use_semantic=False,
    rerank=True,
    rerank_use_graph=False,
    rerank_use_popularity=True,
    over_retrieve_factor=15,
    over_retrieve_cap=200,
    rerank_candidate_cap=200,
)
# Instantiate backend components
vectorstore = QdrantStore(
        model_name="microsoft/codebert-base",
        qdrant_url="http://localhost:6333",
        collection_name="rag_collection_codebert-base_cosine",
        api_key="@lmafa12",
        distance_type="cosine"
    )
kg_retriever = KnowledgeGraphRetriever(vector_store=vectorstore, neo4j_url ="bolt://localhost:7687", neo4j_username="neo4j", neo4j_password ="password", database= "neo4j")
deduplicator = Deduplicator(embedder=vectorstore.embeddings)
reranker = Reranker()

# Build LLM
llm_model = "mistralai/mistral-7b-instruct-v0.3"
llm, tokenizer, gen = build_llm(llm_model,quantize=True,use_8bit=True)
# Instantiate both RAG variants

repo_rag = RepositoryRAG(
    vectorstore,
    retriever=kg_retriever,
    deduplicator=deduplicator,
    reranker=reranker,
    llm=gen,
    neo4j_auth=("neo4j", "password"),
    neo4j_uri="bolt://localhost:7687",
)

No sentence-transformers model found with name microsoft/codebert-base. Creating a new one with mean pooling.
d:\EricssonCodeGraph\hgb-rag-cqa\src\utils\qdrant_store.py:28: UserWarning: Api key is used with an insecure connection.
  self.client = QdrantClient(url=qdrant_url, api_key=api_key)


Successfully connected to Qdrant
Collection 'rag_collection_codebert-base_cosine' already exists.


Device set to use cuda:0
Loading checkpoint shards: 100%|██████████| 3/3 [00:06<00:00,  2.06s/it]
Device set to use cuda:0


In [7]:
from src.eval.evaluation import RAGEvaluator
import pandas as pd
import mlflow
from ast import literal_eval

mlflow.set_tracking_uri("http://127.0.0.1:5000")
experiment_name = "rag_ablation_study"
mlflow.set_experiment(experiment_name)
df = pd.read_csv("data/big_df_filtered.csv")
df["edit_functions"] = df["edit_functions"].apply(literal_eval)
run_name = "exp-2-retr-kg_dedup-True_minhash-True_semdep-False_rr-True_rrgraph-True_rrpop-True_orf-15_orc-200_rcc-200"


evaluator = RAGEvaluator(df.copy(), repo_rag, k_values=[3, 5, 10])
evaluator.evaluate(config, run_name=run_name, verbose=False)
print(f"[{0}] Finished {run_name}")

Evaluating: 100%|██████████| 19/19 [09:24<00:00, 29.69s/item, MRR=0, BLEU=0.02, BERT=-0.002, SemSim=0.454]

🏃 View run exp-2-retr-kg_dedup-True_minhash-True_semdep-False_rr-True_rrgraph-True_rrpop-True_orf-15_orc-200_rcc-200 at: http://127.0.0.1:5000/#/experiments/1/runs/c9d8ad46055e442d9f2d2667ad479a6b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[0] Finished exp-2-retr-kg_dedup-True_minhash-True_semdep-False_rr-True_rrgraph-True_rrpop-True_orf-15_orc-200_rcc-200


In [21]:
df = pd.read_csv("data/eval_prs_nonull_w_metrics.csv")
#df["edit_functions"] = df["edit_functions"].apply(literal_eval)
df

,question,answer,issue_url,edit_functions,problem_statement,comments,pr_problem_statement,pr_comments,precision_3,recall_3,...,recall_10,f1_10,iou_10,mrr,bleu,meteor,bertscore,faithfulness,answer_relevancy,semantic_similarity
0,Why were the `requires_fit` tags set to `False...,The `requires_fit` tags were set to `False` be...,https://github.com/scikit-learn/scikit-learn/p...,"['FeatureHasher.__sklearn_tags__', 'test_featu...",NaN,NaN,Fix requires_fit tag for stateless FeatureHash...,NaN,1.000000,0.333333,...,0.333333,0.500000,0.333333,1.000000,0.159387,0.503567,0.297416,NaN,NaN,0.665623
1,What is the purpose of modifying the error mes...,This improvement fixes a misleading error mess...,https://github.com/scikit-learn/scikit-learn/p...,"['_check_sample_weight_common', '_check_sample...",NaN,NaN,modifying the error message for _check_sample_...,NaN,0.333333,0.500000,...,0.500000,0.166667,0.090909,1.000000,0.019486,0.275309,0.061478,NaN,NaN,0.503138
2,Why did the code change the way files are open...,The code changed the way files are opened and ...,https://github.com/scikit-learn/scikit-learn/p...,"['_fetch_brute_kddcup99', '_load_imgs']",NaN,NaN,FIX: Use context managers to safely close data...,NaN,1.000000,0.500000,...,0.500000,0.666667,0.500000,1.000000,0.010005,0.186624,-0.101393,NaN,NaN,0.311699
3,What is the primary purpose of the DBSCAN algo...,The DBSCAN algorithm is a density-based cluste...,https://github.com/scikit-learn/scikit-learn/p...,"['DBSCAN.fit_predict', 'dbscan']",NaN,NaN,DOC Enhance DBSCAN docstrings with clearer par...,NaN,0.333333,0.500000,...,0.500000,0.166667,0.111111,1.000000,0.099939,0.352950,0.264523,NaN,NaN,0.886228
4,What is the primary function used in the sciki...,The fetch_california_housing function in sklea...,https://github.com/scikit-learn/scikit-learn/p...,"['fetch_california_housing', 'fetch_olivetti_f...",NaN,NaN,MNT Remove redundant mkdir calls\n`get_data_ho...,NaN,0.333333,0.333333,...,0.333333,0.153846,0.090909,1.000000,0.065923,0.407650,0.289067,NaN,NaN,0.786129
5,What is the purpose of the recent changes in _...,"The changes aim to improve the docstring, corr...",https://github.com/scikit-learn/scikit-learn/p...,['_check_array_api_dispatch'],NaN,NaN,MNT Improve _check_array_api_dispatch docstrin...,NaN,0.333333,1.000000,...,1.000000,0.181818,0.100000,0.500000,0.009107,0.191489,0.145026,NaN,NaN,0.594370
6,What is the purpose of adding `array_api_dispa...,It ensures that the assert checks for `device(...,https://github.com/scikit-learn/scikit-learn/p...,"['test_array_api_train_test_split', 'check_arr...",NaN,NaN,TST fix check_array_api_input device check\nTh...,NaN,0.333333,0.333333,...,1.000000,0.461538,0.300000,1.000000,0.014340,0.165094,0.060214,NaN,NaN,0.527707
7,Why did the PR fix the empty column check in C...,The PR fixed the empty column check in ColumnT...,https://github.com/scikit-learn/scikit-learn/p...,"['_is_empty_column_selection', 'test_column_tr...",NaN,NaN,Fix empty column check in ColumnTransformer to...,NaN,1.000000,0.750000,...,1.000000,1.000000,1.000000,1.000000,0.019564,0.125196,-0.026247,NaN,NaN,0.834974
8,How does the `_most_frequent` function handle ...,"The function now ensures safe, deterministic t...",https://github.com/scikit-learn/scikit-learn/p...,['_most_frequent'],NaN,NaN,BUG: Fix _most_frequent to safely handle incom...,NaN,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,0.016714,0.224840,0.010738,NaN,NaN,0.481787
9,Why did the PR convert `test_column_transforme...,To reduce seed sensitivity.,https://github.com/scikit-learn/scikit-learn/p...,['test_column_transformer_auto_memmap'],NaN,NaN,TST use global_random_seed in sklearn/compose/...,NaN,0.333333,1.000000,...,1.000000,0.181818,0.111111,1.000000,0.005433,0.098039,-0.201584,NaN,NaN,0.275048


In [7]:
query = "HOw does pca.fit work?"

top_docs, query_type = repo_rag._retrieve(query, config)

Using KG retriever to fetch top 150 results.


In [8]:
query_type

'general_question'

In [9]:
top_docs

[Document(metadata={'type': 'function_name', 'node_id': 6266, 'doc_id': '1f27e677-cbd2-462d-a345-4038feb1e16d', 'chunk_size': 150, '_id': 'f65d69e8-bf41-4a19-b049-30b9be788066', '_collection_name': 'rag_collection_codebert-base_cosine'}, page_content='DontPickleAttributeMixin.__setstate__'),
 Document(metadata={'type': 'function_name', 'node_id': 6265, 'doc_id': '7507690c-35f5-457c-a9c9-d40d239a3a63', 'chunk_size': 150, '_id': '4a3b4ab1-b57e-4d7a-815e-a04484653024', '_collection_name': 'rag_collection_codebert-base_cosine'}, page_content='DontPickleAttributeMixin.__getstate__'),
 Document(metadata={'type': 'function_name', 'node_id': 5639, 'doc_id': 'ee17390a-27b3-4d71-b4e7-e99cdfcbb4dd', 'chunk_size': 150, '_id': '3b720373-387c-413f-82b7-8bcb32358eb3', '_collection_name': 'rag_collection_codebert-base_cosine'}, page_content='KBinsDiscretizer.fit'),
 Document(metadata={'type': 'function_name', 'node_id': 324, 'doc_id': '3d57ab90-0a61-423f-a093-2aa5f7515ed1', 'chunk_size': 150, '_id': '

In [ ]:
repo_rag.deduplicate()